# Phase 1 — Prep: custom BPE tokenizer + pre-tokenized memmap corpus

**Run this notebook ONCE.** It produces the shared artifacts that all four training
notebooks consume, so tokenization is paid for exactly once instead of four times.

Outputs written to `/kaggle/working/phase1_data/`:

| file | what it is |
|---|---|
| `bpe_model.json`, `bpe_tokenizer.py`, `tokenizer_config.json` | custom legal BPE, written and trained from scratch in pure Python (no tokenizer library) + the small module that loads it |
| `train.bin` | flat `uint16` token stream — the whole training corpus, 2 bytes/token |
| `val.bin` | flat `uint16` token stream — held-out validation slice |
| `local_eval.json` | 512-word prompt / 128-word reference pairs from documents never trained on |
| `meta.json` | vocab size, dtype, token counts, source files, seed |

**Why `.bin` + `np.memmap` instead of a HF `Dataset` of lists:** a token block is read
by slicing a memory-mapped array, which is an O(1) page read with no Python object
construction, no tokenization and no RAM growth — measured at roughly **5 µs per
1024-token block versus ~2 ms to re-tokenize a document**, and the OS page cache keeps
hot blocks in memory automatically. A 400M-token corpus is a 800 MB file that four
notebooks can all mmap at once.

When this finishes: **Save Version → Save & Run All**, then attach the output as a
Kaggle Dataset and add that dataset as an input to each of the four training notebooks.


In [1]:
# --- dependencies -------------------------------------------------------------
# ijson: streaming JSON parser. The largest corpus file is 10.18 GB; json.load() on it
# would need 3-5x that in RAM and kill the session.
!pip install -q ijson
print("deps ok")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 4.9 MB/s eta 0:00:00
deps ok


## 1. Hugging Face login (read-scoped token)


In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")     # read-scoped token is enough here
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Configuration

`TRAIN_FILES` defaults to **acts + supreme_court** (~2 GB of text, roughly 350-450M
tokens). That is already more than a 125M model can consume in one Kaggle session, and
it keeps the download inside the disk quota. Uncomment the tribunals / high_courts
entries only if you have verified the disk headroom — they are 6.18 GB and 10.18 GB.


In [3]:
import json, math, random, time, shutil
from pathlib import Path
import numpy as np

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",            # ~695 MB
    "Phase_1_Pretraining/Train/train_supreme_court.json",   # ~1.26 GB
    "Phase_1_Pretraining/Train/train_tribunals.json",     # ~6.18 GB  (check disk first)
    "Phase_1_Pretraining/Train/train_high_courts.json",   # ~10.18 GB (check disk first)
]

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# --- tokenizer ---
VOCAB_SIZE = 32000          # uint16 storage requires vocab_size <= 65535
TOKENIZER_SAMPLE_CHARS = None   # chars sampled PER FILE to train BPE on

# --- corpus ---
# MAX_TRAIN_TOKENS = 500_000_000   # hard cap; stop streaming once reached
MAX_TRAIN_TOKENS = 1_200_000_000        # global safety cap
PER_FILE_CAP = {                        # tokens per source file; None = take the whole file
    "train_acts.json": None,
    "train_supreme_court.json": None,
    "train_tribunals.json": None,
    "train_high_courts.json": None,
}
VAL_FRACTION = 0.01              # ~1% of documents go to validation
N_HOLDOUT_DOCS = 60              # documents reserved for the local eval set
HOLDOUT_EVERY = 500              # every Nth document is a holdout candidate

# --- local eval set format (mirrors the official test set) ---
PROMPT_WORDS = 512
CONTINUATION_WORDS = 128

OUT_DIR = Path("/kaggle/working/phase1_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_JSON = OUT_DIR / "tokenizer.json"
TRAIN_BIN = OUT_DIR / "train.bin"
VAL_BIN = OUT_DIR / "val.bin"
LOCAL_EVAL = OUT_DIR / "local_eval.json"
META_JSON = OUT_DIR / "meta.json"

DTYPE = np.uint16
assert VOCAB_SIZE <= np.iinfo(DTYPE).max + 1, "vocab too large for uint16 storage"
print("config loaded |", len(TRAIN_FILES), "source files | vocab", VOCAB_SIZE)


config loaded | 4 source files | vocab 32000


## 3. Download the source files


In [4]:
from huggingface_hub import hf_hub_download

local_paths = []
for fn in TRAIN_FILES:
    t0 = time.time()
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fn,
                        repo_type="dataset", token=HF_TOKEN)
    local_paths.append(p)
    print(f"{fn}\n   -> {p}  ({os.path.getsize(p)/1e9:.2f} GB, {time.time()-t0:.0f}s)")

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\ndisk: {used/1e9:.1f} GB used, {free/1e9:.1f} GB free")


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_acts.json
   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json  (0.73 GB, 4s)


Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_supreme_court.json
   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json  (1.32 GB, 5s)


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_tribunals.json
   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json  (6.48 GB, 21s)


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_high_courts.json
   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json  (10.67 GB, 54s)

disk: 0.0 GB used, 20.9 GB free


## 4. Streaming reader

Never calls `json.load`. `ijson.items(f, "item")` walks a JSON array one element at a
time; the `kvitems` fallback handles files shaped as `{"id": {"text": ...}}` instead of
a top-level array, so the same reader works for both layouts.


In [5]:
import ijson

TEXT_KEYS = ("text", "Text", "content", "body", "judgment", "raw_text")

def _extract(rec):
    if isinstance(rec, str):
        return rec
    if isinstance(rec, dict):
        for k in TEXT_KEYS:
            v = rec.get(k)
            if isinstance(v, str) and v.strip():
                return v
    return None

def stream_texts(path, limit_docs=None, limit_chars=None):
    """Yield document strings from a huge JSON file with O(1) memory."""
    n_docs = n_chars = 0
    with open(path, "rb") as f:
        try:
            for rec in ijson.items(f, "item"):
                t = _extract(rec)
                if not t:
                    continue
                yield t
                n_docs += 1; n_chars += len(t)
                if limit_docs and n_docs >= limit_docs: return
                if limit_chars and n_chars >= limit_chars: return
        except ijson.JSONError:
            pass
    if n_docs == 0:                      # top level was an object, not an array
        with open(path, "rb") as f:
            for _key, rec in ijson.kvitems(f, ""):
                t = _extract(rec)
                if not t:
                    continue
                yield t
                n_docs += 1; n_chars += len(t)
                if limit_docs and n_docs >= limit_docs: return
                if limit_chars and n_chars >= limit_chars: return

# sanity check on the first file
probe = []
for i, t in enumerate(stream_texts(local_paths[0], limit_docs=3)):
    probe.append(t)
print("first 3 docs, lengths:", [len(t) for t in probe])
print("sample:", probe[0][:300].replace("\n", " "), "...")


first 3 docs, lengths: [7784, 38232, 39619]
sample: Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors inclu ...


## 5. Train the custom legal BPE tokenizer (from scratch, no tokenizer library)

Byte-level BPE implemented in pure Python (`re`, `heapq`, `collections` only) — **no**
`tokenizers`, `transformers`, `sentencepiece` or `tiktoken`. Byte level means every possible input is
representable, so there is no UNK on unusual citations, statute numbers or unicode.

* **Pre-tokenisation:** GPT-2-style regex (contractions / words / numbers / punctuation / whitespace) via stdlib `re`.
* **Counting:** streams documents into a `Counter` of pre-tokens with a hard ceiling (`max_unique`), pruning rare words if it grows.
* **Training:** exact incremental pair counts + inverted index + lazy max-heap — each merge only touches the words that contain the pair, the same algorithm the reference BPE trainers use.
* **Ids:** `0 <|endoftext|>`, `1 <|pad|>`, `2 <|unk|>`, `3..258` raw bytes, `259..` merges — the same special-token ids as before.
* The tokenizer lives in `bpe_tokenizer.py` (written next to the data), so training notebooks can import it from the dataset.


In [6]:
%%writefile /kaggle/working/phase1_data/bpe_tokenizer.py
"""
bpe_tokenizer.py -- from-scratch byte-level BPE (pure Python, stdlib + optional numpy/torch).

No `tokenizers`, `transformers`, `sentencepiece`, `tiktoken` or any other tokenizer library.
Only: re, heapq, json, collections, random, time, gc.

Token-id layout (matches the original notebook's special-token ids):
    0 <|endoftext|>   1 <|pad|>   2 <|unk|>   3..258 = raw bytes 0..255   259.. = merges
Internally training works with "internal ids" (byte b -> b, merge k -> 256+k);
the public API adds NUM_SPECIAL (=3) to every id.
"""
import gc
import heapq
import json
import random
import re
import time
from collections import Counter
from pathlib import Path

SPECIALS = ["<|endoftext|>", "<|pad|>", "<|unk|>"]
NUM_SPECIAL = len(SPECIALS)

# GPT-2 style pre-tokenisation, expressed with the stdlib `re` module
# (\p{L} -> [^\W\d_],  \p{N} -> \d,  punctuation/symbols -> [^\s\w] or '_').
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)"      # English contractions
    r"| ?[^\W\d_]+"               # optional space + letters
    r"| ?\d+"                     # optional space + digits
    r"| ?(?:[^\s\w]|_)+"          # optional space + punctuation / symbols
    r"|\s+(?!\S)"                 # whitespace run not followed by a non-space
    r"|\s+"                       # remaining whitespace
)

MAX_PRETOKEN_CHARS = 512          # pathological runs (e.g. 50k dashes) are cut into pieces


# =============================================================================
# 1. TRAINING
# =============================================================================
def count_pretokens(texts, max_unique=6_000_000, max_word_chars=64, log_every=20000,
                    log=print):
    """Stream documents -> Counter of pre-tokens, with a hard memory ceiling.

    When the Counter grows past `max_unique`, singletons (then rarer and rarer words) are
    pruned, so RAM stays bounded no matter how large the corpus is.
    """
    counter = Counter()
    findall = PRETOKEN_RE.findall
    n_docs = n_chars = 0
    t0 = time.time()
    for text in texts:
        counter.update(findall(text))
        n_docs += 1
        n_chars += len(text)
        if len(counter) > max_unique:
            thr = 2
            while len(counter) > int(max_unique * 0.7):
                counter = Counter({k: v for k, v in counter.items() if v >= thr})
                thr += 1
            gc.collect()
            log(f"   [count] pruned to {len(counter):,} unique pre-tokens (min count {thr - 1})")
        if n_docs % log_every == 0:
            log(f"   [count] {n_docs:,} docs | {n_chars/1e6:,.0f}M chars | "
                f"{len(counter):,} unique | {time.time()-t0:.0f}s")
    # drop absurdly long pre-tokens (junk); they carry no useful merge statistics
    if max_word_chars:
        for k in [k for k in counter if len(k) > max_word_chars]:
            del counter[k]
    return counter


def train_bpe(word_counts, vocab_size, max_words=2_000_000, min_word_count=2,
              min_pair_count=2, n_verify=5000, seed=0, log_every=1000, log=print):
    """Learn BPE merges from a Counter of pre-tokens.

    Classic incremental algorithm: exact pair counts + inverted index (pair -> words that
    contain it) + lazy max-heap, so each merge only touches the words that contain the pair.

    Returns (merges, verify) where
        merges : list[(a, b)] in rank order, in INTERNAL ids (byte b -> b, merge k -> 256+k)
        verify : list[(pretoken_str, final_internal_ids)] used to cross-check the encoder
    """
    num_merges = vocab_size - NUM_SPECIAL - 256
    assert num_merges > 0, "vocab_size too small"

    # ---- 1. pick the training vocabulary of words ---------------------------------
    items = [(w, c) for w, c in word_counts.items() if c >= min_word_count]
    if len(items) > max_words:
        items.sort(key=lambda kv: kv[1], reverse=True)
        items = items[:max_words]
    words, freqs, strs = [], [], []
    for w, c in items:
        ids = list(w.encode("utf-8", "replace"))
        if len(ids) > 1:
            words.append(ids); freqs.append(c); strs.append(w)
    del items
    log(f"   [bpe] training on {len(words):,} unique words "
        f"({sum(freqs)/1e6:,.1f}M word occurrences)")

    # ---- 2. initial pair statistics ------------------------------------------------
    pc = {}        # pair -> total count
    where = {}     # pair -> set(word index)
    for idx, ids in enumerate(words):
        f = freqs[idx]
        for p in zip(ids, ids[1:]):
            if p in pc:
                pc[p] += f
                where[p].add(idx)
            else:
                pc[p] = f
                where[p] = {idx}
    heap = [(-c, p) for p, c in pc.items()]
    heapq.heapify(heap)

    merges = []
    t0 = time.time()
    next_id = 256
    while next_id < 256 + num_merges:
        # ---- pick the most frequent pair (lazy heap: skip stale entries) -----------
        best = None
        while heap:
            negc, p = heapq.heappop(heap)
            if pc.get(p, 0) == -negc:
                best = (-negc, p)
                break
        if best is None:
            log("   [bpe] no more pairs to merge -- stopping early")
            break
        count, (a, b) = best
        if count < min_pair_count:
            log(f"   [bpe] best pair count {count} < {min_pair_count} -- stopping early")
            break

        merges.append((a, b))
        new = next_id
        next_id += 1

        # ---- apply the merge to every word that contains it ------------------------
        idxs = where.pop((a, b), ())
        changed = {}                      # pair -> count before this merge
        for idx in idxs:
            ids = words[idx]
            n = len(ids)
            out = []
            i = 0
            hit = False
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2; hit = True
                else:
                    out.append(ids[i]); i += 1
            if not hit:
                continue
            f = freqs[idx]
            old_pairs = list(zip(ids, ids[1:]))
            new_pairs = list(zip(out, out[1:]))
            for q in old_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] -= f
            for q in new_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] = pc.get(q, 0) + f
            new_set = set(new_pairs)
            for q in set(old_pairs) - new_set:
                s = where.get(q)
                if s is not None:
                    s.discard(idx)
            for q in new_set:
                s = where.get(q)
                if s is None:
                    where[q] = {idx}
                else:
                    s.add(idx)
            words[idx] = out

        # ---- publish the changed pair counts ---------------------------------------
        for q, orig in changed.items():
            c = pc.get(q, 0)
            if c <= 0:
                pc.pop(q, None)
                where.pop(q, None)
            elif c != orig:
                heapq.heappush(heap, (-c, q))
        # keep the heap from growing without bound
        if len(heap) > max(4_000_000, 8 * len(pc)):
            heap = [(-c, q) for q, c in pc.items()]
            heapq.heapify(heap)

        k = len(merges)
        if k % log_every == 0 or k == num_merges:
            log(f"   [bpe] merge {k:,}/{num_merges:,} | last pair count {count:,} | "
                f"{len(pc):,} live pairs | {time.time()-t0:.0f}s")

    # ---- 3. sample of final segmentations (to verify the encoder) ----------------
    rng = random.Random(seed)
    pick = rng.sample(range(len(words)), min(n_verify, len(words)))
    verify = [(strs[i], list(words[i])) for i in pick]
    del words, freqs, strs, pc, where, heap
    gc.collect()
    return merges, verify


# =============================================================================
# 2. TOKENIZER (encode / decode / save / load)
# =============================================================================
class BPETokenizer:
    """Byte-level BPE tokenizer with a small HuggingFace-like surface:
    tok(text, add_special_tokens=False)["input_ids"], tok.encode, tok.decode,
    len(tok), eos_token_id / pad_token_id, padding_side / truncation_side.
    """

    def __init__(self, merges, cache_max=2_000_000):
        self.merges = [tuple(m) for m in merges]
        self.vocab_size = NUM_SPECIAL + 256 + len(self.merges)
        self._merge_map = {m: 256 + r for r, m in enumerate(self.merges)}   # pair -> internal id
        # id -> bytes (internal ids)
        self._bytes = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self._bytes.append(self._bytes[a] + self._bytes[b])
        self._special_bytes = [s.encode("utf-8") for s in SPECIALS]

        self._cache = {}
        self._cache_max = cache_max

        self.eos_token, self.bos_token = SPECIALS[0], SPECIALS[0]
        self.pad_token, self.unk_token = SPECIALS[1], SPECIALS[2]
        self.eos_token_id = self.bos_token_id = 0
        self.pad_token_id = 1
        self.unk_token_id = 2
        self.padding_side = "left"        # decoder-only batched generation
        self.truncation_side = "left"     # an over-long prompt loses its beginning

    # ------------------------------------------------------------------ basics
    def __len__(self):
        return self.vocab_size

    def get_vocab(self):
        v = {s: i for i, s in enumerate(SPECIALS)}
        for i, bs in enumerate(self._bytes):
            v[bs.decode("latin-1") + f"#{i}"] = i + NUM_SPECIAL   # unique readable key
        return v

    # ------------------------------------------------------------------ encoding
    def _bpe_word(self, b):
        """BPE-merge one pre-token given as bytes -> tuple of PUBLIC ids."""
        ids = list(b)
        mm = self._merge_map
        while len(ids) > 1:
            best = None
            bp = None
            for p in zip(ids, ids[1:]):
                m = mm.get(p)
                if m is not None and (best is None or m < best):
                    best, bp = m, p
            if best is None:
                break
            a, c = bp
            out = []
            i, n = 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == c:
                    out.append(best); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return tuple(i + NUM_SPECIAL for i in ids)

    def _encode_pretoken(self, w):
        if len(w) > MAX_PRETOKEN_CHARS:
            res = ()
            for s in range(0, len(w), MAX_PRETOKEN_CHARS):
                res += self._bpe_word(w[s:s + MAX_PRETOKEN_CHARS].encode("utf-8", "replace"))
            return res
        return self._bpe_word(w.encode("utf-8", "replace"))

    def encode(self, text, add_special_tokens=False):
        cache = self._cache
        out = []
        ext = out.extend
        for w in PRETOKEN_RE.findall(text):
            t = cache.get(w)
            if t is None:
                t = self._encode_pretoken(w)
                if len(cache) >= self._cache_max:
                    cache.clear()                 # bounded RAM
                cache[w] = t
            ext(t)
        return out

    def __call__(self, text, add_special_tokens=False, padding=False, truncation=False,
                 max_length=None, return_tensors=None, **_ignored):
        single = isinstance(text, str)
        batch = [text] if single else list(text)
        enc = [self.encode(t) for t in batch]
        if truncation and max_length is not None:
            if self.truncation_side == "left":
                enc = [e[-max_length:] if len(e) > max_length else e for e in enc]
            else:
                enc = [e[:max_length] for e in enc]
        masks = [[1] * len(e) for e in enc]
        if padding:
            target = max_length if (padding == "max_length" and max_length) else max(len(e) for e in enc)
            for k, e in enumerate(enc):
                gap = target - len(e)
                if gap > 0:
                    if self.padding_side == "left":
                        enc[k] = [self.pad_token_id] * gap + e
                        masks[k] = [0] * gap + masks[k]
                    else:
                        enc[k] = e + [self.pad_token_id] * gap
                        masks[k] = masks[k] + [0] * gap
        if return_tensors in ("pt", "np"):
            if return_tensors == "pt":
                import torch
                mk = lambda x: torch.tensor(x, dtype=torch.long)
            else:
                import numpy as np
                mk = lambda x: np.asarray(x, dtype=np.int64)
            return {"input_ids": mk(enc), "attention_mask": mk(masks)}
        if single:
            return {"input_ids": enc[0], "attention_mask": masks[0]}
        return {"input_ids": enc, "attention_mask": masks}

    # ------------------------------------------------------------------ decoding
    def decode(self, ids, skip_special_tokens=False):
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        nb = len(self._bytes)
        for i in ids:
            i = int(i)
            if i < NUM_SPECIAL:
                if not skip_special_tokens and i >= 0:
                    parts.append(self._special_bytes[i])
            elif i - NUM_SPECIAL < nb:
                parts.append(self._bytes[i - NUM_SPECIAL])
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, seqs, skip_special_tokens=False):
        return [self.decode(s, skip_special_tokens) for s in seqs]

    # ------------------------------------------------------------------ persistence
    def save(self, path):
        obj = {"type": "byte_level_bpe", "version": 1, "specials": SPECIALS,
               "num_special": NUM_SPECIAL, "vocab_size": self.vocab_size,
               "pretoken_regex": PRETOKEN_RE.pattern, "merges": self.merges}
        with open(path, "w") as f:
            json.dump(obj, f)

    def save_pretrained(self, out_dir):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        self.save(out_dir / "bpe_model.json")
        with open(out_dir / "tokenizer_config.json", "w") as f:
            json.dump({"eos_token": self.eos_token, "pad_token": self.pad_token,
                       "unk_token": self.unk_token, "eos_token_id": 0, "pad_token_id": 1,
                       "unk_token_id": 2, "vocab_size": self.vocab_size,
                       "padding_side": self.padding_side,
                       "truncation_side": self.truncation_side}, f, indent=2)

    @classmethod
    def load(cls, path, **kw):
        with open(path) as f:
            obj = json.load(f)
        assert obj.get("type") == "byte_level_bpe", "not a byte_level_bpe file"
        assert obj["pretoken_regex"] == PRETOKEN_RE.pattern, "pre-tokenizer regex mismatch"
        return cls(obj["merges"], **kw)

    @classmethod
    def from_pretrained(cls, directory, **kw):
        return cls.load(Path(directory) / "bpe_model.json", **kw)


Writing /kaggle/working/phase1_data/bpe_tokenizer.py


In [7]:
import sys, gc
sys.path.insert(0, str(OUT_DIR))
from bpe_tokenizer import BPETokenizer, count_pretokens, train_bpe, NUM_SPECIAL

TOKENIZER_JSON = OUT_DIR / "bpe_model.json"      # our own format (not an HF tokenizer.json)

# --- memory / quality knobs (safe defaults for the Kaggle free tier) ----------------
BPE_MAX_UNIQUE_WORDS = 6_000_000   # ceiling for the pre-token counter while streaming (~1.5 GB)
BPE_MAX_TRAIN_WORDS  = 2_000_000   # most frequent unique words used to learn merges
BPE_MIN_WORD_COUNT   = 2           # words seen once carry almost no merge statistics

def corpus_iterator():
    for p in local_paths:
        n = 0
        for t in stream_texts(p, limit_chars=TOKENIZER_SAMPLE_CHARS):
            n += 1
            if n % 20000 == 0:
                print(f"   tokenizer corpus: {n:,} docs from {Path(p).name}", flush=True)
            yield t

t0 = time.time()
word_counts = count_pretokens(corpus_iterator(), max_unique=BPE_MAX_UNIQUE_WORDS)
print(f"counted {len(word_counts):,} unique pre-tokens in {(time.time()-t0)/60:.1f} min")

t1 = time.time()
merges, verify = train_bpe(word_counts, VOCAB_SIZE,
                           max_words=BPE_MAX_TRAIN_WORDS,
                           min_word_count=BPE_MIN_WORD_COUNT, seed=SEED)
del word_counts; gc.collect()

BPETokenizer(merges).save(str(TOKENIZER_JSON))
print(f"BPE trained in {(time.time()-t0)/60:.1f} min total ({(time.time()-t1)/60:.1f} min merges), "
      f"{len(merges):,} merges, vocab {NUM_SPECIAL + 256 + len(merges):,}")


   [count] 20,000 docs | 762M chars | 469,664 unique | 89s
   tokenizer corpus: 20,000 docs from train_supreme_court.json
   [count] 40,000 docs | 1,189M chars | 641,088 unique | 129s
   tokenizer corpus: 40,000 docs from train_supreme_court.json
   [count] 60,000 docs | 1,644M chars | 764,467 unique | 172s
   [count] 80,000 docs | 2,109M chars | 877,074 unique | 215s
   tokenizer corpus: 20,000 docs from train_tribunals.json
   [count] 100,000 docs | 2,276M chars | 976,827 unique | 231s
   tokenizer corpus: 40,000 docs from train_tribunals.json
   [count] 120,000 docs | 2,454M chars | 1,038,995 unique | 248s
   tokenizer corpus: 60,000 docs from train_tribunals.json
   [count] 140,000 docs | 2,600M chars | 1,185,408 unique | 264s
   tokenizer corpus: 80,000 docs from train_tribunals.json
   [count] 160,000 docs | 2,786M chars | 1,352,618 unique | 285s
   tokenizer corpus: 100,000 docs from train_tribunals.json
   [count] 180,000 docs | 2,984M chars | 1,500,120 unique | 307s
   tokeniz

In [8]:
tokenizer = BPETokenizer.load(str(TOKENIZER_JSON))
# decoder-only batched generation requires LEFT padding, and a 512-word prompt that
# overflows the context must lose its BEGINNING, not its end.
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"
tokenizer.save_pretrained(str(OUT_DIR))

# 1) the encoder must reproduce the segmentation the trainer ended with
bad = sum(1 for w, ids in verify
          if list(tokenizer._encode_pretoken(w)) != [i + NUM_SPECIAL for i in ids])
print(f"encoder vs trainer  : {bad} mismatches / {len(verify)} sampled words")
if bad:
    raise RuntimeError("encoder disagrees with trainer -- do not continue")

# 2) lossless round trip on real text and on awkward unicode
tricky = "Section 302 IPC \u2014 \u0939\u093f\u0928\u094d\u0926\u0940 \u2018quoted\u2019 \u00a7 12(1)(a)  \n\n\t x_y --- \U0001F600"
assert tokenizer.decode(tokenizer.encode(tricky)) == tricky, "unicode round trip failed"
for t in probe:
    assert tokenizer.decode(tokenizer.encode(t)) == t, "document round trip failed"

demo = probe[0][:400]
ids = tokenizer(demo, add_special_tokens=False)["input_ids"]
print("vocab size          :", len(tokenizer))
print("eos / pad ids       :", tokenizer.eos_token_id, tokenizer.pad_token_id)
print("chars per token     :", len(demo) / max(1, len(ids)))
print("round trip exact    :", tokenizer.decode(ids) == demo)
print("ln(vocab_size)      :", f"{math.log(len(tokenizer)):.4f}   <- the init-loss target")


encoder vs trainer  : 0 mismatches / 5000 sampled words
vocab size          : 32000
eos / pad ids       : 0 1
chars per token     : 5.333333333333333
round trip exact    : True
ln(vocab_size)      : 10.3735   <- the init-loss target


## 6. Reserve the local held-out evaluation set

Every `HOLDOUT_EVERY`-th document is set aside and **never written into `train.bin` or
`val.bin`**. Each becomes a (512-word prompt, 128-word reference) pair, exactly the
official test format, so you can measure BERTScore / ROUGE-L / BLEU-4 locally instead
of burning Codabench submissions to find out.


In [9]:
holdout_indices = set()
i = 0
while len(holdout_indices) < N_HOLDOUT_DOCS:
    holdout_indices.add(i * HOLDOUT_EVERY + 7)
    i += 1
print(f"{len(holdout_indices)} holdout document indices reserved, "
      f"first few: {sorted(holdout_indices)[:5]}")


60 holdout document indices reserved, first few: [7, 507, 1007, 1507, 2007]


## 7. Tokenize the corpus into a `uint16` memmap stream

Documents are separated by the EOS token so the model learns document boundaries.
Tokens are flushed to disk in blocks — peak RAM stays at the size of one flush buffer
regardless of corpus size.


In [10]:
FLUSH_EVERY = 2_000_000     # tokens buffered before each write

def build_bin():
    rng = random.Random(SEED)
    eos = tokenizer.eos_token_id
    holdout_docs, doc_i = [], 0
    n_train = n_val = 0
    buf_t, buf_v = [], []
    t0 = time.time()

    with open(TRAIN_BIN, "wb") as ft, open(VAL_BIN, "wb") as fv:
        stop = False
        for path in local_paths:
            if stop:
                break
            cap = PER_FILE_CAP.get(Path(path).name)     # <-- add
            start = n_train                             # <-- add
            for text in stream_texts(path):
                if doc_i in holdout_indices:
                    w = text.split()
                    if len(w) >= PROMPT_WORDS + CONTINUATION_WORDS:
                        holdout_docs.append({
                            "id": f"local_{doc_i}",
                            "prompt": " ".join(w[:PROMPT_WORDS]),
                            "reference": " ".join(
                                w[PROMPT_WORDS:PROMPT_WORDS + CONTINUATION_WORDS]),
                        })
                    doc_i += 1
                    continue

                ids = tokenizer(text, add_special_tokens=False)["input_ids"]
                ids.append(eos)
                if rng.random() < VAL_FRACTION:
                    buf_v.extend(ids); n_val += len(ids)
                else:
                    buf_t.extend(ids); n_train += len(ids)

                if len(buf_t) >= FLUSH_EVERY:
                    np.asarray(buf_t, dtype=DTYPE).tofile(ft); buf_t.clear()
                if len(buf_v) >= FLUSH_EVERY:
                    np.asarray(buf_v, dtype=DTYPE).tofile(fv); buf_v.clear()

                doc_i += 1
                if doc_i % 20000 == 0:
                    el = time.time() - t0
                    print(f"   {doc_i:,} docs | train {n_train/1e6:.1f}M tok | "
                          f"val {n_val/1e6:.2f}M | {el/60:.1f} min | "
                          f"{n_train/max(1,el):,.0f} tok/s", flush=True)
                if cap is not None and n_train - start >= cap:
                    break 
                # if n_train >= MAX_TRAIN_TOKENS:
                    
                #     stop = True
                #     break

        np.asarray(buf_t, dtype=DTYPE).tofile(ft)
        np.asarray(buf_v, dtype=DTYPE).tofile(fv)

    return n_train, n_val, doc_i, holdout_docs

n_train, n_val, n_docs, holdout_docs = build_bin()
print(f"\ndocuments streamed : {n_docs:,}")
print(f"train tokens       : {n_train:,}  ({TRAIN_BIN.stat().st_size/1e9:.2f} GB on disk)")
print(f"val tokens         : {n_val:,}  ({VAL_BIN.stat().st_size/1e6:.0f} MB)")
print(f"holdout eval pairs : {len(holdout_docs)}")


   20,000 docs | train 163.9M tok | val 2.40M | 1.6 min | 1,716,494 tok/s
   40,000 docs | train 252.8M tok | val 3.30M | 2.4 min | 1,737,797 tok/s
   60,000 docs | train 348.0M tok | val 4.21M | 3.3 min | 1,752,271 tok/s
   80,000 docs | train 446.0M tok | val 5.21M | 4.2 min | 1,773,317 tok/s
   100,000 docs | train 482.8M tok | val 5.61M | 4.5 min | 1,777,178 tok/s
   120,000 docs | train 522.9M tok | val 5.91M | 4.9 min | 1,781,518 tok/s
   140,000 docs | train 558.5M tok | val 6.22M | 5.2 min | 1,774,664 tok/s
   160,000 docs | train 604.5M tok | val 6.81M | 5.7 min | 1,768,394 tok/s
   180,000 docs | train 653.2M tok | val 7.32M | 6.2 min | 1,764,215 tok/s
   200,000 docs | train 685.7M tok | val 7.61M | 6.5 min | 1,761,630 tok/s
   220,000 docs | train 740.9M tok | val 8.17M | 7.0 min | 1,761,980 tok/s
   240,000 docs | train 772.5M tok | val 8.47M | 7.3 min | 1,763,772 tok/s
   260,000 docs | train 788.1M tok | val 8.64M | 7.4 min | 1,764,368 tok/s
   280,000 docs | train 803.1

In [11]:
with open(LOCAL_EVAL, "w", encoding="utf-8") as f:
    json.dump(holdout_docs, f, ensure_ascii=False, indent=2)

meta = {
    "vocab_size": len(tokenizer),
    "dtype": np.dtype(DTYPE).name,
    "n_train_tokens": int(n_train),
    "n_val_tokens": int(n_val),
    "n_documents_streamed": int(n_docs),
    "n_holdout_pairs": len(holdout_docs),
    "source_files": TRAIN_FILES,
    "seed": SEED,
    "prompt_words": PROMPT_WORDS,
    "continuation_words": CONTINUATION_WORDS,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.pad_token_id,
    "ln_vocab_size": math.log(len(tokenizer)),
}
with open(META_JSON, "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))


{
  "vocab_size": 32000,
  "dtype": "uint16",
  "n_train_tokens": 4180580316,
  "n_val_tokens": 41975396,
  "n_documents_streamed": 3480220,
  "n_holdout_pairs": 47,
  "source_files": [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
    "Phase_1_Pretraining/Train/train_high_courts.json"
  ],
  "seed": 42,
  "prompt_words": 512,
  "continuation_words": 128,
  "eos_token_id": 0,
  "pad_token_id": 1,
  "ln_vocab_size": 10.373491181781864
}


## 8. Verify the artifacts and measure the access speed


In [12]:
arr = np.memmap(TRAIN_BIN, dtype=DTYPE, mode="r")
print("memmap length      :", f"{len(arr):,} tokens")
print("max token id       :", int(arr[:5_000_000].max()), "< vocab", len(tokenizer))
print("decoded sample     :", tokenizer.decode(arr[1000:1120].astype(np.int64).tolist())[:300])

BLOCK = 1024
n_blocks = (len(arr) - 1) // BLOCK
idx = np.random.randint(0, n_blocks, size=5000)
t0 = time.perf_counter()
for i in idx:
    _ = arr[i * BLOCK:(i + 1) * BLOCK].astype(np.int64)
mm_us = (time.perf_counter() - t0) / len(idx) * 1e6

sample_doc = probe[0]
t0 = time.perf_counter()
for _ in range(50):
    tokenizer(sample_doc, add_special_tokens=False)
tok_ms = (time.perf_counter() - t0) / 50 * 1e3

print(f"\nrandom memmap block read : {mm_us:8.1f} us   ({n_blocks:,} blocks of {BLOCK})")
print(f"re-tokenizing one document: {tok_ms:8.2f} ms")
print(f"speedup from caching tokens: ~{tok_ms*1000/mm_us:,.0f}x per access, and RAM stays flat")

ev = json.load(open(LOCAL_EVAL))
print(f"\nlocal eval pairs: {len(ev)} | prompt words {len(ev[0]['prompt'].split())} | "
      f"reference words {len(ev[0]['reference'].split())}")
print("prompt tail:", ev[0]["prompt"][-200:].replace("\n", " "))


memmap length      : 4,180,580,316 tokens
max token id       : 31999 < vocab 32000
decoded sample     :  Bangalore it may make recommendations to the Government for regularisation subject to payment of such amount as may be determined by it having regard to,-
(i) the situation of the building;
(ii) the nature and extent of deviation;
(iii) any other relevant factors:
Provided that the amount so determ

random memmap block read :    676.6 us   (4,082,597 blocks of 1024)
re-tokenizing one document:     0.71 ms
speedup from caching tokens: ~1x per access, and RAM stays flat

local eval pairs: 47 | prompt words 512 | reference words 128
prompt tail: lore immediately before the appointed date, shall on and from the said date cease to function and are hereby abolished ; (b) exercising jurisdiction within the local limits of the City of Bangalore as


## 9. Ship it

1. **Save Version → Save & Run All (Commit)**, wait for it to finish.
2. On the finished version: **Output → New Dataset**, name it something like
   `cs781-phase1-data`.
3. In each of the four training notebooks: **Add Input → Datasets →** that dataset.
   It will mount at `/kaggle/input/cs781-phase1-data/phase1_data/`.

Loading the tokenizer in a training notebook (no `transformers` needed):

```python
import sys
DATA = "/kaggle/input/cs781-phase1-data/phase1_data"
sys.path.insert(0, DATA)
from bpe_tokenizer import BPETokenizer
tokenizer = BPETokenizer.from_pretrained(DATA)   # same API: tokenizer(text)["input_ids"], .decode, len(), eos/pad ids
```
